<a href="https://colab.research.google.com/github/uvindu827/J26-DS-350/blob/Indhi_Component2_CHL_Forecast/02_XGBoost_CHL_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ======================================================================
# STEP 31.9P-13 — XGBOOST MODEL DEVELOPMENT
# PART A — ENVIRONMENT AND GPU VERIFICATION
# ======================================================================

import sys
import platform
import os

print("=" * 70)
print("STEP 31.9P-13 — XGBOOST MODEL DEVELOPMENT")
print("=" * 70)

print("\nPython version:")
print(sys.version)

print("\nPlatform:")
print(platform.platform())

print("\nCUDA_VISIBLE_DEVICES:")
print(os.environ.get("CUDA_VISIBLE_DEVICES"))

# ----------------------------------------------------------------------
# GPU CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("GPU VERIFICATION")
print("=" * 70)

try:
    import torch

    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU device:", torch.cuda.get_device_name(0))
        print("GPU count:", torch.cuda.device_count())
    else:
        print("No CUDA GPU detected through PyTorch.")

except Exception as e:
    print("PyTorch GPU check unavailable:", e)

# ----------------------------------------------------------------------
# XGBOOST VERSION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("XGBOOST VERSION")
print("=" * 70)

import xgboost as xgb

print("XGBoost version:", xgb.__version__)

print("\nEnvironment check complete.")

STEP 31.9P-13 — XGBOOST MODEL DEVELOPMENT

Python version:
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

Platform:
Linux-6.6.122+-x86_64-with-glibc2.39

CUDA_VISIBLE_DEVICES:
None

GPU VERIFICATION
PyTorch version: 2.11.0+cu130
CUDA available: True
GPU device: Tesla T4
GPU count: 1

XGBOOST VERSION
XGBoost version: 3.4.1

Environment check complete.


In [2]:
# ======================================================================
# STEP 31.9P-13 — XGBOOST MODEL DEVELOPMENT
# PART B — PROJECT DIRECTORIES
# ======================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

PREPROCESS_DIR = (
    BASE_DIR /
    "05_model_preprocessing"
)

MODEL_DIR = (
    BASE_DIR /
    "06_models" /
    "02_XGBoost"
)

EVALUATION_DIR = MODEL_DIR / "evaluation"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("STEP 31.9P-13 — PART B")
print("XGBOOST PROJECT DIRECTORIES")
print("=" * 70)

print("\nProject directory:")
print(BASE_DIR)

print("\nPreprocessing directory:")
print(PREPROCESS_DIR)

print("\nXGBoost model directory:")
print(MODEL_DIR)

print("\nEvaluation directory:")
print(EVALUATION_DIR)

print("\n✓ XGBoost directories ready.")

Mounted at /content/drive
STEP 31.9P-13 — PART B
XGBOOST PROJECT DIRECTORIES

Project directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Preprocessing directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing

XGBoost model directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost

Evaluation directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/evaluation

✓ XGBoost directories ready.


In [3]:
# ======================================================================
# STEP 31.9P-13 — XGBOOST MODEL DEVELOPMENT
# PART C — LOAD FINAL MODEL-READY DATA
# ======================================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 31.9P-13 — PART C")
print("LOADING FINAL MODEL-READY DATA")
print("=" * 70)

# ----------------------------------------------------------------------
# FILE PATHS
# ----------------------------------------------------------------------

TRAIN_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_train.csv"
)

VALIDATION_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_validation.csv"
)

TEST_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_test.csv"
)

FEATURE_FILE = (
    PREPROCESS_DIR /
    "final_model_feature_columns.csv"
)

# ----------------------------------------------------------------------
# LOAD DATA
# ----------------------------------------------------------------------

print("\nLoading training data...")
train_df = pd.read_csv(TRAIN_FILE)

print("Loading validation data...")
validation_df = pd.read_csv(VALIDATION_FILE)

print("Loading test data...")
test_df = pd.read_csv(TEST_FILE)

print("Loading final feature list...")
feature_df = pd.read_csv(FEATURE_FILE)

# ----------------------------------------------------------------------
# FEATURE LIST
# ----------------------------------------------------------------------

FEATURES = (
    feature_df["feature"]
    .dropna()
    .astype(str)
    .tolist()
)

# Remove accidental duplicates while preserving original order
FEATURES = list(dict.fromkeys(FEATURES))

# ----------------------------------------------------------------------
# OUTPUT
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL-READY DATA LOADED")
print("=" * 70)

print("\nTraining shape:")
print(train_df.shape)

print("\nValidation shape:")
print(validation_df.shape)

print("\nTest shape:")
print(test_df.shape)

print("\nNumber of selected predictors:")
print(len(FEATURES))

print("\nTarget:")
print("chlorophyll_a_log1p")

print("\nOriginal target:")
print("chlorophyll_a")

# ----------------------------------------------------------------------
# FEATURE EXISTENCE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE EXISTENCE CHECK")
print("=" * 70)

missing_train_features = [
    f for f in FEATURES if f not in train_df.columns
]

missing_validation_features = [
    f for f in FEATURES if f not in validation_df.columns
]

missing_test_features = [
    f for f in FEATURES if f not in test_df.columns
]

print("\nMissing training features:")
print(len(missing_train_features))

print("Missing validation features:")
print(len(missing_validation_features))

print("Missing test features:")
print(len(missing_test_features))

assert len(missing_train_features) == 0
assert len(missing_validation_features) == 0
assert len(missing_test_features) == 0

assert len(FEATURES) == 69

print("\n✓ All 69 selected predictors are available.")
print("✓ Training, validation and test datasets are ready.")

STEP 31.9P-13 — PART C
LOADING FINAL MODEL-READY DATA

Loading training data...
Loading validation data...
Loading test data...
Loading final feature list...

MODEL-READY DATA LOADED

Training shape:
(736950, 75)

Validation shape:
(157896, 75)

Test shape:
(157998, 75)

Number of selected predictors:
69

Target:
chlorophyll_a_log1p

Original target:
chlorophyll_a

FEATURE EXISTENCE CHECK

Missing training features:
0
Missing validation features:
0
Missing test features:
0

✓ All 69 selected predictors are available.
✓ Training, validation and test datasets are ready.


In [4]:
# ======================================================================
# STEP 31.9P-13 — XGBOOST MODEL DEVELOPMENT
# PART D — MODEL INPUT & SAFETY VERIFICATION
# ======================================================================

TARGET = "chlorophyll_a_log1p"
ORIGINAL_TARGET = "chlorophyll_a"

print("=" * 70)
print("STEP 31.9P-13 — PART D")
print("MODEL INPUT & SAFETY VERIFICATION")
print("=" * 70)

# ----------------------------------------------------------------------
# CREATE MODEL INPUTS
# ----------------------------------------------------------------------

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_validation = validation_df[FEATURES]
y_validation = validation_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

# ----------------------------------------------------------------------
# SHAPES
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL INPUT SHAPES")
print("=" * 70)

print("\nX_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nX_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

print("\nX_test:", X_test.shape)
print("y_test:", y_test.shape)

# ----------------------------------------------------------------------
# MISSING VALUE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("MISSING VALUE CHECK")
print("=" * 70)

train_missing = X_train.isna().sum().sum()
validation_missing = X_validation.isna().sum().sum()
test_missing = X_test.isna().sum().sum()

print("\nTraining predictor missing values:", train_missing)
print("Validation predictor missing values:", validation_missing)
print("Test predictor missing values:", test_missing)

print("\nTraining target missing values:", y_train.isna().sum())
print("Validation target missing values:", y_validation.isna().sum())
print("Test target missing values:", y_test.isna().sum())

# ----------------------------------------------------------------------
# INFINITE VALUE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("INFINITE VALUE CHECK")
print("=" * 70)

train_inf = np.isinf(X_train.to_numpy()).sum()
validation_inf = np.isinf(X_validation.to_numpy()).sum()
test_inf = np.isinf(X_test.to_numpy()).sum()

print("\nTraining infinite values:", train_inf)
print("Validation infinite values:", validation_inf)
print("Test infinite values:", test_inf)

# ----------------------------------------------------------------------
# TARGET RANGE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TARGET RANGE CHECK")
print("=" * 70)

print("\nTraining log1p CHL:")
print("Minimum:", y_train.min())
print("Maximum:", y_train.max())

print("\nValidation log1p CHL:")
print("Minimum:", y_validation.min())
print("Maximum:", y_validation.max())

print("\nTest log1p CHL:")
print("Minimum:", y_test.min())
print("Maximum:", y_test.max())

# ----------------------------------------------------------------------
# TARGET LEAKAGE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TARGET LEAKAGE CHECK")
print("=" * 70)

target_in_features = TARGET in FEATURES
original_target_in_features = ORIGINAL_TARGET in FEATURES

print("\nLog target included as predictor:", target_in_features)
print("Original CHL included as predictor:", original_target_in_features)

assert target_in_features is False
assert original_target_in_features is False

print("\n✓ Target is not included among predictors.")

# ----------------------------------------------------------------------
# FEATURE COUNT
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE COUNT CHECK")
print("=" * 70)

print("\nExpected features:", 69)
print("Actual features:", len(FEATURES))

assert len(FEATURES) == 69

# ----------------------------------------------------------------------
# ALL SAFETY ASSERTIONS
# ----------------------------------------------------------------------

assert train_missing == 0
assert validation_missing == 0
assert test_missing == 0

assert y_train.isna().sum() == 0
assert y_validation.isna().sum() == 0
assert y_test.isna().sum() == 0

assert train_inf == 0
assert validation_inf == 0
assert test_inf == 0

assert X_train.shape[1] == 69
assert X_validation.shape[1] == 69
assert X_test.shape[1] == 69

print("\n" + "=" * 70)
print("PART D SAFETY VERIFICATION COMPLETE")
print("=" * 70)

print("\n✓ 69 predictors confirmed.")
print("✓ No missing predictor values.")
print("✓ No missing target values.")
print("✓ No infinite predictor values.")
print("✓ Log CHL target confirmed.")
print("✓ Original CHL target preserved.")
print("✓ No target leakage detected.")
print("✓ Train / validation / test inputs ready.")

STEP 31.9P-13 — PART D
MODEL INPUT & SAFETY VERIFICATION

MODEL INPUT SHAPES

X_train: (736950, 69)
y_train: (736950,)

X_validation: (157896, 69)
y_validation: (157896,)

X_test: (157998, 69)
y_test: (157998,)

MISSING VALUE CHECK

Training predictor missing values: 0
Validation predictor missing values: 0
Test predictor missing values: 0

Training target missing values: 0
Validation target missing values: 0
Test target missing values: 0

INFINITE VALUE CHECK

Training infinite values: 0
Validation infinite values: 0
Test infinite values: 0

TARGET RANGE CHECK

Training log1p CHL:
Minimum: 0.048629258843912
Maximum: 3.5595049524910376

Validation log1p CHL:
Minimum: 0.0646056188864668
Maximum: 3.650702915620501

Test log1p CHL:
Minimum: 0.0535915106642413
Maximum: 3.3375234111374232

TARGET LEAKAGE CHECK

Log target included as predictor: False
Original CHL included as predictor: False

✓ Target is not included among predictors.

FEATURE COUNT CHECK

Expected features: 69
Actual featu